In [1]:
"""
Revision analysis for the G&I major revision (multi-omics AD pipeline).
Run top to bottom. Needs, in DATA_DIR:
  - GSE131617-GPL5175_series_matrix.txt
  - GPL5175-3188.txt                      (platform annotation)
  - Supplementary_Table_S1_significant_SNPs.csv   (GRCh37 positions, from repo)
Downloads the Ensembl GRCh37 GTF (release 87) on first run.
Outputs CSVs to OUT_DIR. Seed 42 everywhere.
"""
import os, re, gzip, urllib.request
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import digamma, polygamma

DATA_DIR = r"C:\Users\a\AlzheimerResearch"
OUT_DIR = os.path.join(DATA_DIR, "revision_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
WINDOW = 50_000
N_PERM = 1000
SEED = 42
# APOE region, GRCh37: APOE (chr19:45,409,011-45,412,650) +/- 1 Mb
APOE_REGION = ("19", 44_409_011, 46_412_650)
GTF_URL = ("https://ftp.ensembl.org/pub/grch37/release-87/gtf/homo_sapiens/"
           "Homo_sapiens.GRCh37.87.gtf.gz")


def P(*a):
    print(*a, flush=True)


# ---------------------------------------------------------------- helpers
def bh(p):
    p = np.asarray(p, float); n = len(p)
    o = np.argsort(p); r = np.empty(n)
    r[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(r, 1)


def trigamma_inv(x, tol=1e-8):
    if x > 1e7: return 1 / np.sqrt(x)
    if x < 1e-6: return 1 / x
    y = 0.5 + 1 / x
    for _ in range(50):
        tri = polygamma(1, y)
        dif = tri * (1 - tri / x) / polygamma(2, y)
        y += dif
        if -dif / y < tol: break
    return y


def fit_fdist(s2, d):
    """Smyth (2004) empirical-Bayes prior (s0^2, d0), as in limma::fitFDist."""
    z = np.log(s2[s2 > 0])
    e = z - digamma(d / 2) + np.log(d / 2)
    emean, evar = e.mean(), e.var(ddof=1) - polygamma(1, d / 2)
    if evar > 0:
        d0 = 2 * trigamma_inv(evar)
        s02 = np.exp(emean + digamma(d0 / 2) - np.log(d0 / 2))
    else:
        d0, s02 = np.inf, np.exp(emean)
    return s02, d0


def moderated_fit(Y, X, coef):
    """Y genes x samples (log2). X samples x p. Returns beta, t_mod, p_mod, t_ord, p_ord, d0."""
    n, p = X.shape; d = n - p
    XtXi = np.linalg.inv(X.T @ X)
    B = Y @ X @ XtXi
    R = Y - B @ X.T
    s2 = (R ** 2).sum(1) / d
    v = XtXi[coef, coef]
    s02, d0 = fit_fdist(s2, d)
    s2m = (d0 * s02 + d * s2) / (d0 + d) if np.isfinite(d0) else np.full_like(s2, s02)
    dm = d0 + d
    t = B[:, coef] / np.sqrt(s2m * v)
    pv = 2 * stats.t.sf(np.abs(t), dm)
    t_o = B[:, coef] / np.sqrt(s2 * v)
    p_o = 2 * stats.t.sf(np.abs(t_o), d)
    return B[:, coef], t, pv, t_o, p_o, d0


def hyper_tail(N, K, n, k):
    """P(X >= k): N universe, K GWAS genes in universe, n DEG genes in universe, k overlap."""
    return stats.hypergeom.sf(k - 1, N, K, n)


# ------------------------------------------------ STEP 1: GRCh37 gene annotation
gtf_path = os.path.join(DATA_DIR, "Homo_sapiens.GRCh37.87.gtf.gz")
if not os.path.exists(gtf_path):
    P("Downloading Ensembl GRCh37 release 87 GTF ...")
    urllib.request.urlretrieve(GTF_URL, gtf_path)
rows = []
with gzip.open(gtf_path, "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        c = line.rstrip("\n").split("\t")
        if c[2] != "gene": continue
        a = c[8]
        name = re.search(r'gene_name "([^"]+)"', a)
        bio = re.search(r'gene_biotype "([^"]+)"', a)
        gid = re.search(r'gene_id "([^"]+)"', a)
        if name:
            rows.append((c[0], int(c[3]), int(c[4]), name.group(1), gid.group(1),
                         bio.group(1) if bio else ""))
genes = pd.DataFrame(rows, columns=["chr", "start", "end", "gene", "gene_id", "biotype"])
valid_symbols = set(genes.gene.str.upper())
P("GRCh37 genes with symbols:", len(genes))

# ------------------------------------------------ STEP 2: SNP -> gene (GRCh37, zero distance inside gene)
snps = pd.read_csv(os.path.join(DATA_DIR, "Supplementary_Table_S1_significant_SNPs.csv"))
snps = snps[["CHR", "BP", "SNP", "A1", "A2", "P"]].drop_duplicates(["SNP", "BP"]).copy()
snps["CHR"] = snps["CHR"].astype(str)
P("Unique significant SNPs:", len(snps), "(GRCh37 coordinates, Jansen 2019)")
assert (snps.loc[snps.SNP == "rs429358", "BP"] == 45411941).all(), "unexpected build"


def map_snps(snps, biotypes=None, window=WINDOW):
    g = genes if biotypes is None else genes[genes.biotype.isin(biotypes)]
    out = []
    for chrom, sub in snps.groupby("CHR"):
        gc = g[g.chr == chrom]
        if gc.empty: continue
        gs, ge = gc.start.values, gc.end.values
        for r in sub.itertuples():
            pos = r.BP
            d = np.where(pos < gs, gs - pos, np.where(pos > ge, pos - ge, 0))
            if (d == 0).any():              # intragenic: assign every gene containing the SNP
                hit = np.where(d == 0)[0]
            else:                           # else nearest gene within window
                m = d.min()
                if m > window: continue
                hit = np.where(d == m)[0]
            for h in hit:
                out.append((r.SNP, chrom, pos, r.P, gc.gene.values[h], gc.biotype.values[h],
                            gs[h], ge[h], int(d[h])))
    return pd.DataFrame(out, columns=["SNP", "CHR", "BP_GRCh37", "P", "gene", "biotype",
                                      "gene_start", "gene_end", "distance"])


snp_gene = map_snps(snps)
snp_gene.to_csv(os.path.join(OUT_DIR, "S1_revised_snp_to_gene_GRCh37.csv"), index=False)
gwas_genes = set(snp_gene.gene)
P("SNPs mapped:", snp_gene.SNP.nunique(), "| GWAS genes:", len(gwas_genes))
for rs in ("rs429358", "rs7412"):
    P(rs, "->", snp_gene[snp_gene.SNP == rs][["gene", "distance"]].values.tolist())
snp_gene_pc = map_snps(snps, biotypes=["protein_coding"])
P("Protein-coding-only GWAS genes:", snp_gene_pc.gene.nunique())

# ------------------------------------------------ STEP 3: probe -> gene (validated, ambiguity explicit)
plat_file = [f for f in os.listdir(DATA_DIR) if "GPL5175" in f and f.endswith(".txt")][0]
with open(os.path.join(DATA_DIR, plat_file), encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()
start = next(i for i, l in enumerate(lines)
             if not l.startswith(("#", "^", "!")) and "gene_assignment" in l)
plat = pd.read_csv(os.path.join(DATA_DIR, plat_file), sep="\t", skiprows=start,
                   low_memory=False, on_bad_lines="skip")
recs = []
for pid, ga in zip(plat["ID"].astype(str), plat["gene_assignment"].astype(str)):
    if ga in ("nan", "---", ""):
        recs.append((pid, None, "no_assignment", "")); continue
    syms = set()
    for entry in ga.split("///"):
        parts = [x.strip() for x in entry.split("//")]
        if len(parts) > 1 and parts[1].upper() in valid_symbols:   # symbol field only
            syms.add(parts[1].upper())
    if len(syms) == 1:
        recs.append((pid, syms.pop(), "mapped", ""))
    elif len(syms) == 0:
        recs.append((pid, None, "no_valid_symbol", ga[:120]))
    else:
        recs.append((pid, None, "ambiguous_multi_gene", "|".join(sorted(syms))))
pmap = pd.DataFrame(recs, columns=["probe", "gene", "status", "note"])
pmap.to_csv(os.path.join(OUT_DIR, "probe_to_gene_mapping_table.csv"), index=False)
P(pmap.status.value_counts().to_string())
probe2gene = dict(zip(pmap.probe, pmap.gene))
cyto = [g for g in pmap.gene.dropna().unique() if re.fullmatch(r"\d+[QP]\d+.*", g)]
P("Cytoband-like symbols remaining (must be 0):", len(cyto))

# ------------------------------------------------ STEP 4: expression + metadata (EC only)
gse = os.path.join(DATA_DIR, "GSE131617-GPL5175_series_matrix.txt")
gsm, titles, chars = [], [], []
with open(gse) as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"): gsm = re.findall(r'"(GSM\d+)"', line)
        elif line.startswith("!Sample_title"): titles = re.findall(r'"([^"]*)"', line)
        elif line.startswith("!Sample_characteristics_ch1"):
            chars.append(re.findall(r'"([^"]*)"', line))
        elif line.startswith('"ID_REF"') or line.startswith("ID_REF"): break
meta = pd.DataFrame({"gsm": gsm, "title": titles})
for row in chars:
    keys = {x.split(":")[0].strip().lower() for x in row if ":" in x}
    if len(keys) == 1:
        k = keys.pop()
        meta[k] = [x.split(":", 1)[1].strip() if ":" in x else None for x in row]
P("Metadata columns available:", [c for c in meta.columns if c not in ("gsm", "title")])
sexcol = next((c for c in meta.columns if re.search(r"sex|gender", c)), None)
assert sexcol, "no sex/gender field found in series matrix"
agecols = [c for c in meta.columns if re.search(r"age|pmi|rin|postmortem", c)]
P("Age/PMI/RIN-like fields found:", agecols or "none")
meta = meta[meta.title.str.contains("EC")].copy()
braak_map = {"0": 0, "I-II": 1, "III-IV": 2, "V-VI": 3}
meta["braak"] = meta["braak nft stage"].str.strip().map(braak_map)
assert meta.braak.notna().all(), meta["braak nft stage"].unique()
meta["sex01"] = (meta[sexcol].str.lower().str.startswith("m")).astype(int)
meta.to_csv(os.path.join(OUT_DIR, "sample_level_metadata_EC.csv"), index=False)
P("EC samples by Braak:", meta.braak.value_counts().sort_index().to_dict())

mat = pd.read_csv(gse, sep="\t", comment="!", header=0).set_index("ID_REF")
mat.index = mat.index.astype(str)
probes_tested = mat.index.tolist()
P("Probes tested:", len(probes_tested))
gene_of_probe = pd.Series([probe2gene.get(p) for p in probes_tested], index=probes_tested)
universe = set(gene_of_probe.dropna())
P("Universe = validly mapped, tested genes:", len(universe))

# primary comparison: Braak 0 vs V-VI, EC
sub = meta[meta.braak.isin([0, 3])].copy()
sub["grp"] = (sub.braak == 3).astype(int)
Y = mat[sub.gsm].values.astype(float)              # ORIGINAL log2 values (not row z-scored:
                                                   # z-scoring would erase the variance information
                                                   # that empirical-Bayes moderation borrows)
sex = sub.sex01.values; grp = sub.grp.values
P("Group x sex:\n", pd.crosstab(sub.grp, sub.sex01).to_string())
X = np.column_stack([np.ones(len(sub)), grp, sex])
beta, t_mod, p_mod, t_ord, p_ord, d0 = moderated_fit(Y, X, 1)
P(f"Prior df d0 = {d0:.2f}")
X0 = np.column_stack([np.ones(len(sub)), grp])      # unadjusted, for comparison
b0, t0m, p0m, t0o, p0o, d00 = moderated_fit(Y, X0, 1)
de = pd.DataFrame({"probe": probes_tested, "gene": gene_of_probe.values,
                   "log2FC_adj": beta, "t_mod_adj": t_mod, "p_mod_adj": p_mod,
                   "padj_BH_mod_adj": bh(p_mod), "p_ord_unadj_ttest": p0o,
                   "log2FC_unadj": b0, "p_mod_unadj": p0m})
de.to_csv(os.path.join(OUT_DIR, "S3_revised_probe_level_DE.csv"), index=False)
P("BH<0.05 probes (adjusted moderated):", int((de.padj_BH_mod_adj < 0.05).sum()),
  "| min BH:", de.padj_BH_mod_adj.min())
P("BH<0.05 probes (unadjusted ordinary t, Welch-equivalent pooled):",
  int((bh(p0o) < 0.05).sum()))


def deg_genes(pvals, mask=None):
    m = (pvals < 0.05) & gene_of_probe.notna().values
    return set(gene_of_probe.values[m])


# ------------------------------------------------ STEP 5: overlap tables + hypergeometric
def overlap_report(label, gset, dset):
    g = gset & universe; d = dset & universe; k = g & d
    N, K, n = len(universe), len(g), len(d)
    p = hyper_tail(N, K, n, len(k))
    P(f"[{label}] contingency: universe N={N}, GWAS in universe K={K}, DEG n={n}, "
      f"overlap k={len(k)}, expected={K*n/N:.2f}, hypergeom P(X>=k)={p:.4f}")
    return dict(label=label, N=N, K=K, n=n, k=len(k), expected=K * n / N, p=p,
                genes=",".join(sorted(k)))


deg_adj = deg_genes(p_mod); deg_unadj = deg_genes(p0o)
rep = [overlap_report("unadjusted t-test (original definition), GRCh37-corrected", gwas_genes, deg_unadj),
       overlap_report("moderated + sex-adjusted", gwas_genes, deg_adj)]
# APOE-region exclusion (SNPs in region removed before gene mapping)
inreg = (snp_gene.CHR == APOE_REGION[0]) & snp_gene.BP_GRCh37.between(*APOE_REGION[1:])
gwas_noapoe = set(snp_gene[~inreg].gene) - set(snp_gene[inreg].gene)
rep.append(overlap_report("unadjusted, APOE region excluded", gwas_noapoe, deg_unadj))
rep.append(overlap_report("adjusted, APOE region excluded", gwas_noapoe, deg_adj))
rep.append(overlap_report("adjusted, protein-coding GWAS genes only",
                          set(snp_gene_pc.gene), deg_adj))
pd.DataFrame(rep).to_csv(os.path.join(OUT_DIR, "overlap_hypergeometric_table.csv"), index=False)
cand = sorted((gwas_genes & universe) & deg_adj)
cand_unadj = sorted((gwas_genes & universe) & deg_unadj)
P("Candidates (adjusted):", cand)
P("Candidates (unadjusted, corrected mapping):", cand_unadj)

# ------------------------------------------------ STEP 6: label permutation within sex strata (adjusted model)
rng = np.random.default_rng(SEED)
gidx_all = pd.factorize(gene_of_probe)[0]            # -1 for unmapped
gnames = pd.factorize(gene_of_probe)[1]
mask_all = np.isin(gnames, list(gwas_genes & universe))
mask_noapoe = np.isin(gnames, list(gwas_noapoe & universe))
perm_all, perm_noapoe = [], []
for _ in range(N_PERM):
    gp = grp.copy()
    for s in (0, 1):
        idx = np.where(sex == s)[0]
        gp[idx] = rng.permutation(grp[idx])
    Xp = np.column_stack([np.ones(len(sub)), gp, sex])
    _, _, pp, _, _, _ = moderated_fit(Y, Xp, 1)
    sig = (pp < 0.05) & (gidx_all >= 0)
    gi = np.unique(gidx_all[sig])
    perm_all.append(mask_all[gi].sum()); perm_noapoe.append(mask_noapoe[gi].sum())
for lab, perm, obs in (("all GWAS genes", perm_all, len(gwas_genes & deg_adj & universe)),
                       ("APOE region excluded", perm_noapoe,
                        len(gwas_noapoe & deg_adj & universe))):
    perm = np.array(perm)
    pe = (1 + (perm >= obs).sum()) / (N_PERM + 1)
    P(f"[perm, sex-stratified, adjusted] {lab}: observed={obs}, mean perm={perm.mean():.2f}, "
      f"max={perm.max()}, empirical p={pe:.4f}")

# ------------------------------------------------ STEP 7: candidate statistics table (R3-6, R4-2, R4-5)
gene_level = (de.dropna(subset=["gene"]).sort_values("p_mod_adj")
              .drop_duplicates("gene").set_index("gene"))
ctrl = sub[sub.grp == 0].gsm; ad = sub[sub.grp == 1].gsm
rows = []
for g in sorted(set(cand) | set(cand_unadj)):
    pr = de[(de.gene == g)].sort_values("p_mod_adj").iloc[0]
    v = mat.loc[pr.probe]
    U, pu = stats.mannwhitneyu(v[ad], v[ctrl], alternative="two-sided")
    snpsg = snp_gene[snp_gene.gene == g]
    lead = snpsg.sort_values("P").iloc[0]
    rows.append(dict(gene=g, probe=pr.probe, log2FC_unadj=pr.log2FC_unadj,
                     p_ttest=pr.p_ord_unadj_ttest, log2FC_adj=pr.log2FC_adj,
                     p_mod_adj=pr.p_mod_adj, BH_adj=pr.padj_BH_mod_adj,
                     MW_U=U, MW_p=pu, n_supporting_SNPs=len(snpsg),
                     lead_SNP=lead.SNP, lead_SNP_P=lead.P, lead_chr=lead.CHR,
                     lead_pos_GRCh37=lead.BP_GRCh37, min_distance=snpsg.distance.min(),
                     in_both_lists_adj=g in cand, in_both_lists_unadj=g in cand_unadj))
cs = pd.DataFrame(rows)
cs.to_csv(os.path.join(OUT_DIR, "candidate_statistics_table.csv"), index=False)
P(cs.round(4).to_string())

# ------------------------------------------------ STEP 8: locus grouping (position only; NOT LD)
cl = cs[["gene", "lead_chr", "lead_pos_GRCh37"]].sort_values(["lead_chr", "lead_pos_GRCh37"])
cl["locus"] = (cl.groupby("lead_chr").lead_pos_GRCh37.diff().fillna(0) > 500_000).cumsum()
P("Regions (lead SNPs >500 kb apart = separate region):\n", cl.to_string())

# ------------------------------------------------ STEP 9: threshold-free test (what intersection adds)
gl = de.dropna(subset=["gene"]).sort_values("p_mod_adj").drop_duplicates("gene").set_index("gene")
inG = gl.index.isin(gwas_genes); inN = gl.index.isin(gwas_noapoe)
for lab, m in (("all GWAS genes", inG), ("APOE region excluded", inN)):
    u, pu = stats.mannwhitneyu(gl.p_mod_adj[m], gl.p_mod_adj[~m], alternative="less")
    P(f"[competitive, threshold-free] {lab}: n={m.sum()} median p {np.median(gl.p_mod_adj[m]):.3f} "
      f"vs {np.median(gl.p_mod_adj[~m]):.3f}; one-sided MWU p={pu:.4f}")

# ------------------------------------------------ STEP 10: all EC samples, Braak ordinal (R4-4)
alls = meta.copy()
Ya = mat[alls.gsm].values.astype(float)
Xa = np.column_stack([np.ones(len(alls)), alls.braak.values, alls.sex01.values])
ba, ta, pa, _, _, _ = moderated_fit(Ya, Xa, 1)
dea = pd.DataFrame({"probe": probes_tested, "gene": gene_of_probe.values, "slope_per_braak_step": ba,
                    "t": ta, "p": pa, "BH": bh(pa)})
dea.to_csv(os.path.join(OUT_DIR, "S5_all_EC_samples_braak_ordinal.csv"), index=False)
P(f"All-EC model: n={len(alls)} samples; BH<0.05 probes = {(dea.BH < 0.05).sum()}")
P("Candidates in all-EC model (p, slope):\n",
  dea[dea.gene.isin(cand)].sort_values("p")[["gene", "slope_per_braak_step", "p"]].round(4).to_string())

# ------------------------------------------------ STEP 11: ranked gene-set analysis (prerank)
import gseapy as gp
rank = gl["t_mod_adj"].sort_values(ascending=False)
rank.index = rank.index.astype(str)
for lib in ("KEGG_2021_Human", "GO_Biological_Process_2023"):
    try:
        pre = gp.prerank(rnk=rank, gene_sets=lib, min_size=10, max_size=500,
                         permutation_num=1000, seed=SEED, outdir=None, no_plot=True)
        r = pre.res2d.sort_values("FDR q-val").head(15)
        r.to_csv(os.path.join(OUT_DIR, f"prerank_{lib}.csv"), index=False)
        P(lib, "\n", r[["Term", "NES", "NOM p-val", "FDR q-val", "Lead_genes"]].to_string())
    except Exception as e:
        P("prerank failed for", lib, e)

# ------------------------------------------------ STEP 12: enrichment with explicit background (R1-6, R3-5)
for label, gl_ in (("candidates", cand), ("all adjusted DEGs", sorted(deg_adj & universe))):
    try:
        e = gp.enrichr(gene_list=gl_, gene_sets=["KEGG_2021_Human", "GO_Biological_Process_2023"],
                       background=sorted(universe), outdir=None)
        res = e.results.sort_values("Adjusted P-value")
        res.to_csv(os.path.join(OUT_DIR, f"enrichr_{label.replace(' ', '_')}_bg_tested_genes.csv"),
                   index=False)
        P(label, "- top terms (background = tested, validly mapped genes):\n",
          res[["Gene_set", "Term", "Overlap", "P-value", "Adjusted P-value", "Genes"]]
          .head(10).to_string())
    except Exception as ex:
        P("enrichr failed:", ex)
P("DONE. Outputs in", OUT_DIR)

C:\Users\a\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


GRCh37 genes with symbols: 57905
Unique significant SNPs: 2394 (GRCh37 coordinates, Jansen 2019)
SNPs mapped: 2375 | GWAS genes: 128
rs429358 -> [['APOE', 0]]
rs7412 -> [['APOE', 0]]
Protein-coding-only GWAS genes: 97
status
no_assignment           283444
mapped                   23810
no_valid_symbol           5753
ambiguous_multi_gene      3912
Cytoband-like symbols remaining (must be 0): 0
Metadata columns available: ['subject id', 'tissue', 'tissue region', 'sex', 'braak nft stage']
Age/PMI/RIN-like fields found: ['braak nft stage']
EC samples by Braak: {0: 13, 1: 20, 2: 19, 3: 19}
Probes tested: 22011
Universe = validly mapped, tested genes: 14172
Group x sex:
 sex01   0  1
grp         
0       4  9
1      13  6
Prior df d0 = 3.59
BH<0.05 probes (adjusted moderated): 0 | min BH: 0.07450813587949892
BH<0.05 probes (unadjusted ordinary t, Welch-equivalent pooled): 0
[unadjusted t-test (original definition), GRCh37-corrected] contingency: universe N=14172, GWAS in universe K=60, DEG 

GO_Biological_Process_2023 
                                                                                                   Term       NES NOM p-val FDR q-val                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Lead_genes
1                                        Regulation Of Neurotransmitter Receptor Activity (GO:0099601) -2.200078       0.0  0.014703                                                                                                                                                                                                                                  

enrichr failed: "['Overlap'] not in index"
enrichr failed: "['Overlap'] not in index"
DONE. Outputs in C:\Users\a\AlzheimerResearch\revision_outputs


In [2]:
"""
Revision analysis for the G&I major revision (multi-omics AD pipeline).
Run top to bottom. Needs, in DATA_DIR:
  - GSE131617-GPL5175_series_matrix.txt
  - GPL5175-3188.txt                      (platform annotation)
  - Supplementary_Table_S1_significant_SNPs.csv   (GRCh37 positions, from repo)
Downloads the Ensembl GRCh37 GTF (release 87) on first run.
Outputs CSVs to OUT_DIR. Seed 42 everywhere.
"""
import os, re, gzip, urllib.request
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import digamma, polygamma

DATA_DIR = r"C:\Users\a\AlzheimerResearch"
OUT_DIR = os.path.join(DATA_DIR, "revision_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
WINDOW = 50_000
N_PERM = 1000
SEED = 42
# APOE region, GRCh37: APOE (chr19:45,409,011-45,412,650) +/- 1 Mb
APOE_REGION = ("19", 44_409_011, 46_412_650)
GTF_URL = ("https://ftp.ensembl.org/pub/grch37/release-87/gtf/homo_sapiens/"
           "Homo_sapiens.GRCh37.87.gtf.gz")


def P(*a):
    print(*a, flush=True)


# ---------------------------------------------------------------- helpers
def bh(p):
    p = np.asarray(p, float); n = len(p)
    o = np.argsort(p); r = np.empty(n)
    r[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(r, 1)


def trigamma_inv(x, tol=1e-8):
    if x > 1e7: return 1 / np.sqrt(x)
    if x < 1e-6: return 1 / x
    y = 0.5 + 1 / x
    for _ in range(50):
        tri = polygamma(1, y)
        dif = tri * (1 - tri / x) / polygamma(2, y)
        y += dif
        if -dif / y < tol: break
    return y


def fit_fdist(s2, d):
    """Smyth (2004) empirical-Bayes prior (s0^2, d0), as in limma::fitFDist."""
    z = np.log(s2[s2 > 0])
    e = z - digamma(d / 2) + np.log(d / 2)
    emean, evar = e.mean(), e.var(ddof=1) - polygamma(1, d / 2)
    if evar > 0:
        d0 = 2 * trigamma_inv(evar)
        s02 = np.exp(emean + digamma(d0 / 2) - np.log(d0 / 2))
    else:
        d0, s02 = np.inf, np.exp(emean)
    return s02, d0


def moderated_fit(Y, X, coef):
    """Y genes x samples (log2). X samples x p. Returns beta, t_mod, p_mod, t_ord, p_ord, d0."""
    n, p = X.shape; d = n - p
    XtXi = np.linalg.inv(X.T @ X)
    B = Y @ X @ XtXi
    R = Y - B @ X.T
    s2 = (R ** 2).sum(1) / d
    v = XtXi[coef, coef]
    s02, d0 = fit_fdist(s2, d)
    s2m = (d0 * s02 + d * s2) / (d0 + d) if np.isfinite(d0) else np.full_like(s2, s02)
    dm = d0 + d
    t = B[:, coef] / np.sqrt(s2m * v)
    pv = 2 * stats.t.sf(np.abs(t), dm)
    t_o = B[:, coef] / np.sqrt(s2 * v)
    p_o = 2 * stats.t.sf(np.abs(t_o), d)
    return B[:, coef], t, pv, t_o, p_o, d0


def hyper_tail(N, K, n, k):
    """P(X >= k): N universe, K GWAS genes in universe, n DEG genes in universe, k overlap."""
    return stats.hypergeom.sf(k - 1, N, K, n)


# ------------------------------------------------ STEP 1: GRCh37 gene annotation
gtf_path = os.path.join(DATA_DIR, "Homo_sapiens.GRCh37.87.gtf.gz")
if not os.path.exists(gtf_path):
    P("Downloading Ensembl GRCh37 release 87 GTF ...")
    urllib.request.urlretrieve(GTF_URL, gtf_path)
rows = []
with gzip.open(gtf_path, "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        c = line.rstrip("\n").split("\t")
        if c[2] != "gene": continue
        a = c[8]
        name = re.search(r'gene_name "([^"]+)"', a)
        bio = re.search(r'gene_biotype "([^"]+)"', a)
        gid = re.search(r'gene_id "([^"]+)"', a)
        if name:
            rows.append((c[0], int(c[3]), int(c[4]), name.group(1), gid.group(1),
                         bio.group(1) if bio else ""))
genes = pd.DataFrame(rows, columns=["chr", "start", "end", "gene", "gene_id", "biotype"])
valid_symbols = set(genes.gene.str.upper())
P("GRCh37 genes with symbols:", len(genes))

# ------------------------------------------------ STEP 2: SNP -> gene (GRCh37, zero distance inside gene)
snps = pd.read_csv(os.path.join(DATA_DIR, "Supplementary_Table_S1_significant_SNPs.csv"))
snps = snps[["CHR", "BP", "SNP", "A1", "A2", "P"]].drop_duplicates(["SNP", "BP"]).copy()
snps["CHR"] = snps["CHR"].astype(str)
P("Unique significant SNPs:", len(snps), "(GRCh37 coordinates, Jansen 2019)")
assert (snps.loc[snps.SNP == "rs429358", "BP"] == 45411941).all(), "unexpected build"


def map_snps(snps, biotypes=None, window=WINDOW):
    g = genes if biotypes is None else genes[genes.biotype.isin(biotypes)]
    out = []
    for chrom, sub in snps.groupby("CHR"):
        gc = g[g.chr == chrom]
        if gc.empty: continue
        gs, ge = gc.start.values, gc.end.values
        for r in sub.itertuples():
            pos = r.BP
            d = np.where(pos < gs, gs - pos, np.where(pos > ge, pos - ge, 0))
            if (d == 0).any():              # intragenic: assign every gene containing the SNP
                hit = np.where(d == 0)[0]
            else:                           # else nearest gene within window
                m = d.min()
                if m > window: continue
                hit = np.where(d == m)[0]
            for h in hit:
                out.append((r.SNP, chrom, pos, r.P, gc.gene.values[h], gc.biotype.values[h],
                            gs[h], ge[h], int(d[h])))
    return pd.DataFrame(out, columns=["SNP", "CHR", "BP_GRCh37", "P", "gene", "biotype",
                                      "gene_start", "gene_end", "distance"])


snp_gene = map_snps(snps)
snp_gene.to_csv(os.path.join(OUT_DIR, "S1_revised_snp_to_gene_GRCh37.csv"), index=False)
gwas_genes = set(snp_gene.gene)
P("SNPs mapped:", snp_gene.SNP.nunique(), "| GWAS genes:", len(gwas_genes))
for rs in ("rs429358", "rs7412"):
    P(rs, "->", snp_gene[snp_gene.SNP == rs][["gene", "distance"]].values.tolist())
snp_gene_pc = map_snps(snps, biotypes=["protein_coding"])
P("Protein-coding-only GWAS genes:", snp_gene_pc.gene.nunique())

# ------------------------------------------------ STEP 3: probe -> gene (validated, ambiguity explicit)
plat_file = [f for f in os.listdir(DATA_DIR) if "GPL5175" in f and f.endswith(".txt")][0]
with open(os.path.join(DATA_DIR, plat_file), encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()
start = next(i for i, l in enumerate(lines)
             if not l.startswith(("#", "^", "!")) and "gene_assignment" in l)
plat = pd.read_csv(os.path.join(DATA_DIR, plat_file), sep="\t", skiprows=start,
                   low_memory=False, on_bad_lines="skip")
recs = []
for pid, ga in zip(plat["ID"].astype(str), plat["gene_assignment"].astype(str)):
    if ga in ("nan", "---", ""):
        recs.append((pid, None, "no_assignment", "")); continue
    syms = set()
    for entry in ga.split("///"):
        parts = [x.strip() for x in entry.split("//")]
        if len(parts) > 1 and parts[1].upper() in valid_symbols:   # symbol field only
            syms.add(parts[1].upper())
    if len(syms) == 1:
        recs.append((pid, syms.pop(), "mapped", ""))
    elif len(syms) == 0:
        recs.append((pid, None, "no_valid_symbol", ga[:120]))
    else:
        recs.append((pid, None, "ambiguous_multi_gene", "|".join(sorted(syms))))
pmap = pd.DataFrame(recs, columns=["probe", "gene", "status", "note"])
pmap.to_csv(os.path.join(OUT_DIR, "probe_to_gene_mapping_table.csv"), index=False)
P(pmap.status.value_counts().to_string())
probe2gene = dict(zip(pmap.probe, pmap.gene))
cyto = [g for g in pmap.gene.dropna().unique() if re.fullmatch(r"\d+[QP]\d+.*", g)]
P("Cytoband-like symbols remaining (must be 0):", len(cyto))

# ------------------------------------------------ STEP 4: expression + metadata (EC only)
gse = os.path.join(DATA_DIR, "GSE131617-GPL5175_series_matrix.txt")
gsm, titles, chars = [], [], []
with open(gse) as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"): gsm = re.findall(r'"(GSM\d+)"', line)
        elif line.startswith("!Sample_title"): titles = re.findall(r'"([^"]*)"', line)
        elif line.startswith("!Sample_characteristics_ch1"):
            chars.append(re.findall(r'"([^"]*)"', line))
        elif line.startswith('"ID_REF"') or line.startswith("ID_REF"): break
meta = pd.DataFrame({"gsm": gsm, "title": titles})
for row in chars:
    keys = {x.split(":")[0].strip().lower() for x in row if ":" in x}
    if len(keys) == 1:
        k = keys.pop()
        meta[k] = [x.split(":", 1)[1].strip() if ":" in x else None for x in row]
P("Metadata columns available:", [c for c in meta.columns if c not in ("gsm", "title")])
sexcol = next((c for c in meta.columns if re.search(r"sex|gender", c)), None)
assert sexcol, "no sex/gender field found in series matrix"
agecols = [c for c in meta.columns if re.search(r"age|pmi|rin|postmortem", c)]
P("Age/PMI/RIN-like fields found:", agecols or "none")
meta = meta[meta.title.str.contains("EC")].copy()
braak_map = {"0": 0, "I-II": 1, "III-IV": 2, "V-VI": 3}
meta["braak"] = meta["braak nft stage"].str.strip().map(braak_map)
assert meta.braak.notna().all(), meta["braak nft stage"].unique()
meta["sex01"] = (meta[sexcol].str.lower().str.startswith("m")).astype(int)
meta.to_csv(os.path.join(OUT_DIR, "sample_level_metadata_EC.csv"), index=False)
P("EC samples by Braak:", meta.braak.value_counts().sort_index().to_dict())

mat = pd.read_csv(gse, sep="\t", comment="!", header=0).set_index("ID_REF")
mat.index = mat.index.astype(str)
probes_tested = mat.index.tolist()
P("Probes tested:", len(probes_tested))
gene_of_probe = pd.Series([probe2gene.get(p) for p in probes_tested], index=probes_tested)
universe = set(gene_of_probe.dropna())
P("Universe = validly mapped, tested genes:", len(universe))

# primary comparison: Braak 0 vs V-VI, EC
sub = meta[meta.braak.isin([0, 3])].copy()
sub["grp"] = (sub.braak == 3).astype(int)
Y = mat[sub.gsm].values.astype(float)              # ORIGINAL log2 values (not row z-scored:
                                                   # z-scoring would erase the variance information
                                                   # that empirical-Bayes moderation borrows)
sex = sub.sex01.values; grp = sub.grp.values
P("Group x sex:\n", pd.crosstab(sub.grp, sub.sex01).to_string())
X = np.column_stack([np.ones(len(sub)), grp, sex])
beta, t_mod, p_mod, t_ord, p_ord, d0 = moderated_fit(Y, X, 1)
P(f"Prior df d0 = {d0:.2f}")
X0 = np.column_stack([np.ones(len(sub)), grp])      # unadjusted, for comparison
b0, t0m, p0m, t0o, p0o, d00 = moderated_fit(Y, X0, 1)
de = pd.DataFrame({"probe": probes_tested, "gene": gene_of_probe.values,
                   "log2FC_adj": beta, "t_mod_adj": t_mod, "p_mod_adj": p_mod,
                   "padj_BH_mod_adj": bh(p_mod), "p_ord_unadj_ttest": p0o,
                   "log2FC_unadj": b0, "p_mod_unadj": p0m})
de.to_csv(os.path.join(OUT_DIR, "S3_revised_probe_level_DE.csv"), index=False)
P("BH<0.05 probes (adjusted moderated):", int((de.padj_BH_mod_adj < 0.05).sum()),
  "| min BH:", de.padj_BH_mod_adj.min())
P("BH<0.05 probes (unadjusted ordinary t, Welch-equivalent pooled):",
  int((bh(p0o) < 0.05).sum()))


def deg_genes(pvals, mask=None):
    m = (pvals < 0.05) & gene_of_probe.notna().values
    return set(gene_of_probe.values[m])


# ------------------------------------------------ STEP 5: overlap tables + hypergeometric
def overlap_report(label, gset, dset):
    g = gset & universe; d = dset & universe; k = g & d
    N, K, n = len(universe), len(g), len(d)
    p = hyper_tail(N, K, n, len(k))
    P(f"[{label}] contingency: universe N={N}, GWAS in universe K={K}, DEG n={n}, "
      f"overlap k={len(k)}, expected={K*n/N:.2f}, hypergeom P(X>=k)={p:.4f}")
    return dict(label=label, N=N, K=K, n=n, k=len(k), expected=K * n / N, p=p,
                genes=",".join(sorted(k)))


deg_adj = deg_genes(p_mod); deg_unadj = deg_genes(p0o)
rep = [overlap_report("unadjusted t-test (original definition), GRCh37-corrected", gwas_genes, deg_unadj),
       overlap_report("moderated + sex-adjusted", gwas_genes, deg_adj)]
# APOE-region exclusion (SNPs in region removed before gene mapping)
inreg = (snp_gene.CHR == APOE_REGION[0]) & snp_gene.BP_GRCh37.between(*APOE_REGION[1:])
gwas_noapoe = set(snp_gene[~inreg].gene) - set(snp_gene[inreg].gene)
rep.append(overlap_report("unadjusted, APOE region excluded", gwas_noapoe, deg_unadj))
rep.append(overlap_report("adjusted, APOE region excluded", gwas_noapoe, deg_adj))
rep.append(overlap_report("adjusted, protein-coding GWAS genes only",
                          set(snp_gene_pc.gene), deg_adj))
pd.DataFrame(rep).to_csv(os.path.join(OUT_DIR, "overlap_hypergeometric_table.csv"), index=False)
cand = sorted((gwas_genes & universe) & deg_adj)
cand_unadj = sorted((gwas_genes & universe) & deg_unadj)
P("Candidates (adjusted):", cand)
P("Candidates (unadjusted, corrected mapping):", cand_unadj)

# ------------------------------------------------ STEP 6: label permutation within sex strata (adjusted model)
rng = np.random.default_rng(SEED)
gidx_all = pd.factorize(gene_of_probe)[0]            # -1 for unmapped
gnames = pd.factorize(gene_of_probe)[1]
mask_all = np.isin(gnames, list(gwas_genes & universe))
mask_noapoe = np.isin(gnames, list(gwas_noapoe & universe))
perm_all, perm_noapoe = [], []
for _ in range(N_PERM):
    gp = grp.copy()
    for s in (0, 1):
        idx = np.where(sex == s)[0]
        gp[idx] = rng.permutation(grp[idx])
    Xp = np.column_stack([np.ones(len(sub)), gp, sex])
    _, _, pp, _, _, _ = moderated_fit(Y, Xp, 1)
    sig = (pp < 0.05) & (gidx_all >= 0)
    gi = np.unique(gidx_all[sig])
    perm_all.append(mask_all[gi].sum()); perm_noapoe.append(mask_noapoe[gi].sum())
for lab, perm, obs in (("all GWAS genes", perm_all, len(gwas_genes & deg_adj & universe)),
                       ("APOE region excluded", perm_noapoe,
                        len(gwas_noapoe & deg_adj & universe))):
    perm = np.array(perm)
    pe = (1 + (perm >= obs).sum()) / (N_PERM + 1)
    P(f"[perm, sex-stratified, adjusted] {lab}: observed={obs}, mean perm={perm.mean():.2f}, "
      f"max={perm.max()}, empirical p={pe:.4f}")

# ------------------------------------------------ STEP 7: candidate statistics table (R3-6, R4-2, R4-5)
gene_level = (de.dropna(subset=["gene"]).sort_values("p_mod_adj")
              .drop_duplicates("gene").set_index("gene"))
ctrl = sub[sub.grp == 0].gsm; ad = sub[sub.grp == 1].gsm
rows = []
for g in sorted(set(cand) | set(cand_unadj)):
    pr = de[(de.gene == g)].sort_values("p_mod_adj").iloc[0]
    v = mat.loc[pr.probe]
    U, pu = stats.mannwhitneyu(v[ad], v[ctrl], alternative="two-sided")
    snpsg = snp_gene[snp_gene.gene == g]
    lead = snpsg.sort_values("P").iloc[0]
    rows.append(dict(gene=g, probe=pr.probe, log2FC_unadj=pr.log2FC_unadj,
                     p_ttest=pr.p_ord_unadj_ttest, log2FC_adj=pr.log2FC_adj,
                     p_mod_adj=pr.p_mod_adj, BH_adj=pr.padj_BH_mod_adj,
                     MW_U=U, MW_p=pu, n_supporting_SNPs=len(snpsg),
                     lead_SNP=lead.SNP, lead_SNP_P=lead.P, lead_chr=lead.CHR,
                     lead_pos_GRCh37=lead.BP_GRCh37, min_distance=snpsg.distance.min(),
                     in_both_lists_adj=g in cand, in_both_lists_unadj=g in cand_unadj))
cs = pd.DataFrame(rows)
cs.to_csv(os.path.join(OUT_DIR, "candidate_statistics_table.csv"), index=False)
P(cs.round(4).to_string())

# ------------------------------------------------ STEP 8: locus grouping (position only; NOT LD)
cl = cs[["gene", "lead_chr", "lead_pos_GRCh37"]].sort_values(["lead_chr", "lead_pos_GRCh37"])
cl = cl.reset_index(drop=True)
cl["locus"] = ((cl.lead_chr != cl.lead_chr.shift()) | (cl.lead_pos_GRCh37.diff() > 500_000)).cumsum()
P("Regions (lead SNPs >500 kb apart = separate region):\n", cl.to_string())

# ------------------------------------------------ STEP 9: threshold-free test (what intersection adds)
gl = de.dropna(subset=["gene"]).sort_values("p_mod_adj").drop_duplicates("gene").set_index("gene")
inG = gl.index.isin(gwas_genes); inN = gl.index.isin(gwas_noapoe)
for lab, m in (("all GWAS genes", inG), ("APOE region excluded", inN)):
    u, pu = stats.mannwhitneyu(gl.p_mod_adj[m], gl.p_mod_adj[~m], alternative="less")
    P(f"[competitive, threshold-free] {lab}: n={m.sum()} median p {np.median(gl.p_mod_adj[m]):.3f} "
      f"vs {np.median(gl.p_mod_adj[~m]):.3f}; one-sided MWU p={pu:.4f}")

# ------------------------------------------------ STEP 10: all EC samples, Braak ordinal (R4-4)
alls = meta.copy()
Ya = mat[alls.gsm].values.astype(float)
Xa = np.column_stack([np.ones(len(alls)), alls.braak.values, alls.sex01.values])
ba, ta, pa, _, _, _ = moderated_fit(Ya, Xa, 1)
dea = pd.DataFrame({"probe": probes_tested, "gene": gene_of_probe.values, "slope_per_braak_step": ba,
                    "t": ta, "p": pa, "BH": bh(pa)})
dea.to_csv(os.path.join(OUT_DIR, "S5_all_EC_samples_braak_ordinal.csv"), index=False)
P(f"All-EC model: n={len(alls)} samples; BH<0.05 probes = {(dea.BH < 0.05).sum()}")
P("Candidates in all-EC model (p, slope):\n",
  dea[dea.gene.isin(cand)].sort_values("p")[["gene", "slope_per_braak_step", "p"]].round(4).to_string())

# ------------------------------------------------ STEP 11: ranked gene-set analysis (prerank)
import gseapy as gp
rank = gl["t_mod_adj"].sort_values(ascending=False)
rank.index = rank.index.astype(str)
for lib in ("KEGG_2021_Human", "GO_Biological_Process_2023"):
    try:
        pre = gp.prerank(rnk=rank, gene_sets=lib, min_size=10, max_size=500,
                         permutation_num=1000, seed=SEED, outdir=None, no_plot=True)
        full = pre.res2d.sort_values("FDR q-val")
        full.to_csv(os.path.join(OUT_DIR, f"prerank_{lib}_FULL.csv"), index=False)
        r = full.head(15)
        P(lib, "\n", r[["Term", "NES", "NOM p-val", "FDR q-val"]].to_string())
    except Exception as e:
        P("prerank failed for", lib, e)

# ------------------------------------------------ STEP 12: enrichment with explicit background (R1-6, R3-5)
for label, gl_ in (("candidates", cand), ("all adjusted DEGs", sorted(deg_adj & universe))):
    for bg_label, bg in (("tested_genes_background", sorted(universe)), ("default_background", None)):
        try:
            kw = dict(gene_list=gl_, gene_sets=["KEGG_2021_Human", "GO_Biological_Process_2023"], outdir=None)
            if bg is not None: kw["background"] = bg
            res = gp.enrichr(**kw).results.sort_values("Adjusted P-value")
            res.to_csv(os.path.join(OUT_DIR, f"enrichr_{label.replace(' ', '_')}_{bg_label}.csv"), index=False)
            cols = [c for c in ["Gene_set", "Term", "Overlap", "P-value", "Adjusted P-value", "Genes"]
                    if c in res.columns]
            P(label, bg_label, "- columns:", list(res.columns))
            P(res[cols].head(10).to_string())
        except Exception as ex:
            P("enrichr failed:", label, bg_label, repr(ex))
P("DONE. Outputs in", OUT_DIR)

GRCh37 genes with symbols: 57905
Unique significant SNPs: 2394 (GRCh37 coordinates, Jansen 2019)
SNPs mapped: 2375 | GWAS genes: 128
rs429358 -> [['APOE', 0]]
rs7412 -> [['APOE', 0]]
Protein-coding-only GWAS genes: 97
status
no_assignment           283444
mapped                   23810
no_valid_symbol           5753
ambiguous_multi_gene      3912
Cytoband-like symbols remaining (must be 0): 0
Metadata columns available: ['subject id', 'tissue', 'tissue region', 'sex', 'braak nft stage']
Age/PMI/RIN-like fields found: ['braak nft stage']
EC samples by Braak: {0: 13, 1: 20, 2: 19, 3: 19}
Probes tested: 22011
Universe = validly mapped, tested genes: 14172
Group x sex:
 sex01   0  1
grp         
0       4  9
1      13  6
Prior df d0 = 3.59
BH<0.05 probes (adjusted moderated): 0 | min BH: 0.07450813587949892
BH<0.05 probes (unadjusted ordinary t, Welch-equivalent pooled): 0
[unadjusted t-test (original definition), GRCh37-corrected] contingency: universe N=14172, GWAS in universe K=60, DEG 

candidates default_background - columns: ['Gene_set', 'Term', 'Overlap', 'P-value', 'Adjusted P-value', 'Old P-value', 'Old Adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes']
                      Gene_set                                                                                 Term Overlap   P-value  Adjusted P-value    Genes
1   GO_Biological_Process_2023                                                   Enamel Mineralization (GO:0070166)     1/5  0.000750          0.015745  SLC24A4
2   GO_Biological_Process_2023                                                    Tooth Mineralization (GO:0034505)     1/7  0.001050          0.015745  SLC24A4
3   GO_Biological_Process_2023                                          Regulation Of Feeding Behavior (GO:0060259)    1/14  0.002099          0.020232  SLC24A4
4   GO_Biological_Process_2023                                                 Membrane Repolarization (GO:0086009)    1/18  0.002698          0.020232  SLC24A4
15  GO_Biol

DONE. Outputs in C:\Users\a\AlzheimerResearch\revision_outputs


In [3]:
"""
Revision analysis for the G&I major revision (multi-omics AD pipeline).
Run top to bottom. Needs, in DATA_DIR:
  - GSE131617-GPL5175_series_matrix.txt
  - GPL5175-3188.txt                      (platform annotation)
  - Supplementary_Table_S1_significant_SNPs.csv   (GRCh37 positions, from repo)
Downloads the Ensembl GRCh37 GTF (release 87) on first run.
Outputs CSVs to OUT_DIR. Seed 42 everywhere.
"""
import os, re, gzip, urllib.request
import numpy as np, pandas as pd
from scipy import stats
from scipy.special import digamma, polygamma

DATA_DIR = r"C:\Users\a\AlzheimerResearch"
OUT_DIR = os.path.join(DATA_DIR, "revision_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
WINDOW = 50_000
N_PERM = 1000
SEED = 42
# APOE region, GRCh37: APOE (chr19:45,409,011-45,412,650) +/- 1 Mb
APOE_REGION = ("19", 44_409_011, 46_412_650)
GTF_URL = ("https://ftp.ensembl.org/pub/grch37/release-87/gtf/homo_sapiens/"
           "Homo_sapiens.GRCh37.87.gtf.gz")


def P(*a):
    print(*a, flush=True)


# ---------------------------------------------------------------- helpers
def bh(p):
    p = np.asarray(p, float); n = len(p)
    o = np.argsort(p); r = np.empty(n)
    r[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(r, 1)


def trigamma_inv(x, tol=1e-8):
    if x > 1e7: return 1 / np.sqrt(x)
    if x < 1e-6: return 1 / x
    y = 0.5 + 1 / x
    for _ in range(50):
        tri = polygamma(1, y)
        dif = tri * (1 - tri / x) / polygamma(2, y)
        y += dif
        if -dif / y < tol: break
    return y


def fit_fdist(s2, d):
    """Smyth (2004) empirical-Bayes prior (s0^2, d0), as in limma::fitFDist."""
    z = np.log(s2[s2 > 0])
    e = z - digamma(d / 2) + np.log(d / 2)
    emean, evar = e.mean(), e.var(ddof=1) - polygamma(1, d / 2)
    if evar > 0:
        d0 = 2 * trigamma_inv(evar)
        s02 = np.exp(emean + digamma(d0 / 2) - np.log(d0 / 2))
    else:
        d0, s02 = np.inf, np.exp(emean)
    return s02, d0


def moderated_fit(Y, X, coef):
    """Y genes x samples (log2). X samples x p. Returns beta, t_mod, p_mod, t_ord, p_ord, d0."""
    n, p = X.shape; d = n - p
    XtXi = np.linalg.inv(X.T @ X)
    B = Y @ X @ XtXi
    R = Y - B @ X.T
    s2 = (R ** 2).sum(1) / d
    v = XtXi[coef, coef]
    s02, d0 = fit_fdist(s2, d)
    s2m = (d0 * s02 + d * s2) / (d0 + d) if np.isfinite(d0) else np.full_like(s2, s02)
    dm = d0 + d
    t = B[:, coef] / np.sqrt(s2m * v)
    pv = 2 * stats.t.sf(np.abs(t), dm)
    t_o = B[:, coef] / np.sqrt(s2 * v)
    p_o = 2 * stats.t.sf(np.abs(t_o), d)
    return B[:, coef], t, pv, t_o, p_o, d0


def hyper_tail(N, K, n, k):
    """P(X >= k): N universe, K GWAS genes in universe, n DEG genes in universe, k overlap."""
    return stats.hypergeom.sf(k - 1, N, K, n)


# ------------------------------------------------ STEP 1: GRCh37 gene annotation
gtf_path = os.path.join(DATA_DIR, "Homo_sapiens.GRCh37.87.gtf.gz")
if not os.path.exists(gtf_path):
    P("Downloading Ensembl GRCh37 release 87 GTF ...")
    urllib.request.urlretrieve(GTF_URL, gtf_path)
rows = []
with gzip.open(gtf_path, "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        c = line.rstrip("\n").split("\t")
        if c[2] != "gene": continue
        a = c[8]
        name = re.search(r'gene_name "([^"]+)"', a)
        bio = re.search(r'gene_biotype "([^"]+)"', a)
        gid = re.search(r'gene_id "([^"]+)"', a)
        if name:
            rows.append((c[0], int(c[3]), int(c[4]), name.group(1), gid.group(1),
                         bio.group(1) if bio else ""))
genes = pd.DataFrame(rows, columns=["chr", "start", "end", "gene", "gene_id", "biotype"])
valid_symbols = set(genes.gene.str.upper())
P("GRCh37 genes with symbols:", len(genes))

# ------------------------------------------------ STEP 2: SNP -> gene (GRCh37, zero distance inside gene)
snps = pd.read_csv(os.path.join(DATA_DIR, "Supplementary_Table_S1_significant_SNPs.csv"))
snps = snps[["CHR", "BP", "SNP", "A1", "A2", "P"]].drop_duplicates(["SNP", "BP"]).copy()
snps["CHR"] = snps["CHR"].astype(str)
P("Unique significant SNPs:", len(snps), "(GRCh37 coordinates, Jansen 2019)")
assert (snps.loc[snps.SNP == "rs429358", "BP"] == 45411941).all(), "unexpected build"


def map_snps(snps, biotypes=None, window=WINDOW):
    g = genes if biotypes is None else genes[genes.biotype.isin(biotypes)]
    out = []
    for chrom, sub in snps.groupby("CHR"):
        gc = g[g.chr == chrom]
        if gc.empty: continue
        gs, ge = gc.start.values, gc.end.values
        for r in sub.itertuples():
            pos = r.BP
            d = np.where(pos < gs, gs - pos, np.where(pos > ge, pos - ge, 0))
            if (d == 0).any():              # intragenic: assign every gene containing the SNP
                hit = np.where(d == 0)[0]
            else:                           # else nearest gene within window
                m = d.min()
                if m > window: continue
                hit = np.where(d == m)[0]
            for h in hit:
                out.append((r.SNP, chrom, pos, r.P, gc.gene.values[h], gc.biotype.values[h],
                            gs[h], ge[h], int(d[h])))
    return pd.DataFrame(out, columns=["SNP", "CHR", "BP_GRCh37", "P", "gene", "biotype",
                                      "gene_start", "gene_end", "distance"])


snp_gene = map_snps(snps)
snp_gene.to_csv(os.path.join(OUT_DIR, "S1_revised_snp_to_gene_GRCh37.csv"), index=False)
gwas_genes = set(snp_gene.gene)
P("SNPs mapped:", snp_gene.SNP.nunique(), "| GWAS genes:", len(gwas_genes))
for rs in ("rs429358", "rs7412"):
    P(rs, "->", snp_gene[snp_gene.SNP == rs][["gene", "distance"]].values.tolist())
snp_gene_pc = map_snps(snps, biotypes=["protein_coding"])
P("Protein-coding-only GWAS genes:", snp_gene_pc.gene.nunique())
n_intra = snp_gene.loc[snp_gene.distance == 0, "SNP"].nunique()
n_multi = int((snp_gene.groupby("SNP").gene.nunique() > 1).sum())
P(f"SNPs mapped inside a gene (distance 0): {n_intra}; mapped to nearest gene within window: "
  f"{snp_gene.SNP.nunique() - n_intra}; SNPs assigned to >1 overlapping gene: {n_multi}")
P("Biotypes of the GWAS genes:\n", snp_gene.drop_duplicates("gene").biotype.value_counts().to_string())

# ------------------------------------------------ STEP 3: probe -> gene (validated, ambiguity explicit)
plat_file = [f for f in os.listdir(DATA_DIR) if "GPL5175" in f and f.endswith(".txt")][0]
with open(os.path.join(DATA_DIR, plat_file), encoding="utf-8", errors="ignore") as f:
    lines = f.readlines()
start = next(i for i, l in enumerate(lines)
             if not l.startswith(("#", "^", "!")) and "gene_assignment" in l)
plat = pd.read_csv(os.path.join(DATA_DIR, plat_file), sep="\t", skiprows=start,
                   low_memory=False, on_bad_lines="skip")
recs = []
for pid, ga in zip(plat["ID"].astype(str), plat["gene_assignment"].astype(str)):
    if ga in ("nan", "---", ""):
        recs.append((pid, None, "no_assignment", "")); continue
    syms = set()
    for entry in ga.split("///"):
        parts = [x.strip() for x in entry.split("//")]
        if len(parts) > 1 and parts[1].upper() in valid_symbols:   # symbol field only
            syms.add(parts[1].upper())
    if len(syms) == 1:
        recs.append((pid, syms.pop(), "mapped", ""))
    elif len(syms) == 0:
        recs.append((pid, None, "no_valid_symbol", ga[:120]))
    else:
        recs.append((pid, None, "ambiguous_multi_gene", "|".join(sorted(syms))))
pmap = pd.DataFrame(recs, columns=["probe", "gene", "status", "note"])
pmap.to_csv(os.path.join(OUT_DIR, "probe_to_gene_mapping_table.csv"), index=False)
P(pmap.status.value_counts().to_string())
probe2gene = dict(zip(pmap.probe, pmap.gene))
cyto = [g for g in pmap.gene.dropna().unique() if re.fullmatch(r"\d+[QP]\d+.*", g)]
P("Cytoband-like symbols remaining (must be 0):", len(cyto))

# ------------------------------------------------ STEP 4: expression + metadata (EC only)
gse = os.path.join(DATA_DIR, "GSE131617-GPL5175_series_matrix.txt")
gsm, titles, chars = [], [], []
with open(gse) as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"): gsm = re.findall(r'"(GSM\d+)"', line)
        elif line.startswith("!Sample_title"): titles = re.findall(r'"([^"]*)"', line)
        elif line.startswith("!Sample_characteristics_ch1"):
            chars.append(re.findall(r'"([^"]*)"', line))
        elif line.startswith('"ID_REF"') or line.startswith("ID_REF"): break
meta = pd.DataFrame({"gsm": gsm, "title": titles})
for row in chars:
    keys = {x.split(":")[0].strip().lower() for x in row if ":" in x}
    if len(keys) == 1:
        k = keys.pop()
        meta[k] = [x.split(":", 1)[1].strip() if ":" in x else None for x in row]
P("Metadata columns available:", [c for c in meta.columns if c not in ("gsm", "title")])
sexcol = next((c for c in meta.columns if re.search(r"sex|gender", c)), None)
assert sexcol, "no sex/gender field found in series matrix"
agecols = [c for c in meta.columns if re.search(r"age|pmi|rin|postmortem", c)]
P("Age/PMI/RIN-like fields found:", agecols or "none")
meta = meta[meta.title.str.contains("EC")].copy()
braak_map = {"0": 0, "I-II": 1, "III-IV": 2, "V-VI": 3}
meta["braak"] = meta["braak nft stage"].str.strip().map(braak_map)
assert meta.braak.notna().all(), meta["braak nft stage"].unique()
meta["sex01"] = (meta[sexcol].str.lower().str.startswith("m")).astype(int)
meta.to_csv(os.path.join(OUT_DIR, "sample_level_metadata_EC.csv"), index=False)
P("EC samples by Braak:", meta.braak.value_counts().sort_index().to_dict())

mat = pd.read_csv(gse, sep="\t", comment="!", header=0).set_index("ID_REF")
mat.index = mat.index.astype(str)
probes_tested = mat.index.tolist()
P("Probes tested:", len(probes_tested))
gene_of_probe = pd.Series([probe2gene.get(p) for p in probes_tested], index=probes_tested)
universe = set(gene_of_probe.dropna())
P("Universe = validly mapped, tested genes:", len(universe))

# primary comparison: Braak 0 vs V-VI, EC
sub = meta[meta.braak.isin([0, 3])].copy()
sub["grp"] = (sub.braak == 3).astype(int)
Y = mat[sub.gsm].values.astype(float)              # ORIGINAL log2 values (not row z-scored:
                                                   # z-scoring would erase the variance information
                                                   # that empirical-Bayes moderation borrows)
sex = sub.sex01.values; grp = sub.grp.values
P("Group x sex:\n", pd.crosstab(sub.grp, sub.sex01).to_string())
X = np.column_stack([np.ones(len(sub)), grp, sex])
beta, t_mod, p_mod, t_ord, p_ord, d0 = moderated_fit(Y, X, 1)
P(f"Prior df d0 = {d0:.2f}")
X0 = np.column_stack([np.ones(len(sub)), grp])      # unadjusted, for comparison
b0, t0m, p0m, t0o, p0o, d00 = moderated_fit(Y, X0, 1)
de = pd.DataFrame({"probe": probes_tested, "gene": gene_of_probe.values,
                   "log2FC_adj": beta, "t_mod_adj": t_mod, "p_mod_adj": p_mod,
                   "padj_BH_mod_adj": bh(p_mod), "p_ord_unadj_ttest": p0o,
                   "log2FC_unadj": b0, "p_mod_unadj": p0m})
de.to_csv(os.path.join(OUT_DIR, "S3_revised_probe_level_DE.csv"), index=False)
P("BH<0.05 probes (adjusted moderated):", int((de.padj_BH_mod_adj < 0.05).sum()),
  "| min BH:", de.padj_BH_mod_adj.min())
P("BH<0.05 probes (unadjusted ordinary t, Welch-equivalent pooled):",
  int((bh(p0o) < 0.05).sum()))


def deg_genes(pvals, mask=None):
    m = (pvals < 0.05) & gene_of_probe.notna().values
    return set(gene_of_probe.values[m])


# ------------------------------------------------ STEP 5: overlap tables + hypergeometric
def overlap_report(label, gset, dset):
    g = gset & universe; d = dset & universe; k = g & d
    N, K, n = len(universe), len(g), len(d)
    p = hyper_tail(N, K, n, len(k))
    P(f"[{label}] contingency: universe N={N}, GWAS in universe K={K}, DEG n={n}, "
      f"overlap k={len(k)}, expected={K*n/N:.2f}, hypergeom P(X>=k)={p:.4f}")
    return dict(label=label, N=N, K=K, n=n, k=len(k), expected=K * n / N, p=p,
                genes=",".join(sorted(k)))


deg_adj = deg_genes(p_mod); deg_unadj = deg_genes(p0o)
rep = [overlap_report("unadjusted t-test (original definition), GRCh37-corrected", gwas_genes, deg_unadj),
       overlap_report("moderated + sex-adjusted", gwas_genes, deg_adj)]
# APOE-region exclusion (SNPs in region removed before gene mapping)
inreg = (snp_gene.CHR == APOE_REGION[0]) & snp_gene.BP_GRCh37.between(*APOE_REGION[1:])
gwas_noapoe = set(snp_gene[~inreg].gene) - set(snp_gene[inreg].gene)
rep.append(overlap_report("unadjusted, APOE region excluded", gwas_noapoe, deg_unadj))
rep.append(overlap_report("adjusted, APOE region excluded", gwas_noapoe, deg_adj))
rep.append(overlap_report("adjusted, protein-coding GWAS genes only",
                          set(snp_gene_pc.gene), deg_adj))
pd.DataFrame(rep).to_csv(os.path.join(OUT_DIR, "overlap_hypergeometric_table.csv"), index=False)
cand = sorted((gwas_genes & universe) & deg_adj)
cand_unadj = sorted((gwas_genes & universe) & deg_unadj)
P("Candidates (adjusted):", cand)
P("Candidates (unadjusted, corrected mapping):", cand_unadj)

# ------------------------------------------------ STEP 5b: mapping-window sensitivity (corrected mapping)
wrows = []
for w in (20_000, 50_000, 100_000, 200_000):
    sg = map_snps(snps, window=w)
    gset = set(sg.gene)
    for lab, dset in (("adjusted", deg_adj), ("unadjusted", deg_unadj)):
        g = gset & universe; d_ = dset & universe; k = g & d_
        wrows.append(dict(window_kb=w // 1000, model=lab, GWAS_genes=len(gset), GWAS_in_universe=len(g),
                          overlap=len(k), expected=len(g) * len(d_) / len(universe),
                          hypergeom_p=hyper_tail(len(universe), len(g), len(d_), len(k)),
                          genes=",".join(sorted(k))))
wdf = pd.DataFrame(wrows)
wdf.to_csv(os.path.join(OUT_DIR, "window_sensitivity_table.csv"), index=False)
P("Window sensitivity:\n", wdf.round(4).to_string())
import sys, scipy, numpy, pandas
P("VERSIONS python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__,
  "| scipy", scipy.__version__)
try:
    import gseapy as _g; P("gseapy", _g.__version__)
except Exception: pass

# ------------------------------------------------ STEP 6: label permutation within sex strata (adjusted model)
rng = np.random.default_rng(SEED)
gidx_all = pd.factorize(gene_of_probe)[0]            # -1 for unmapped
gnames = pd.factorize(gene_of_probe)[1]
mask_all = np.isin(gnames, list(gwas_genes & universe))
mask_noapoe = np.isin(gnames, list(gwas_noapoe & universe))
perm_all, perm_noapoe = [], []
for _ in range(N_PERM):
    gp = grp.copy()
    for s in (0, 1):
        idx = np.where(sex == s)[0]
        gp[idx] = rng.permutation(grp[idx])
    Xp = np.column_stack([np.ones(len(sub)), gp, sex])
    _, _, pp, _, _, _ = moderated_fit(Y, Xp, 1)
    sig = (pp < 0.05) & (gidx_all >= 0)
    gi = np.unique(gidx_all[sig])
    perm_all.append(mask_all[gi].sum()); perm_noapoe.append(mask_noapoe[gi].sum())
for lab, perm, obs in (("all GWAS genes", perm_all, len(gwas_genes & deg_adj & universe)),
                       ("APOE region excluded", perm_noapoe,
                        len(gwas_noapoe & deg_adj & universe))):
    perm = np.array(perm)
    pe = (1 + (perm >= obs).sum()) / (N_PERM + 1)
    P(f"[perm, sex-stratified, adjusted] {lab}: observed={obs}, mean perm={perm.mean():.2f}, "
      f"max={perm.max()}, empirical p={pe:.4f}")

# ------------------------------------------------ STEP 7: candidate statistics table (R3-6, R4-2, R4-5)
gene_level = (de.dropna(subset=["gene"]).sort_values("p_mod_adj")
              .drop_duplicates("gene").set_index("gene"))
ctrl = sub[sub.grp == 0].gsm; ad = sub[sub.grp == 1].gsm
rows = []
for g in sorted(set(cand) | set(cand_unadj)):
    pr = de[(de.gene == g)].sort_values("p_mod_adj").iloc[0]
    v = mat.loc[pr.probe]
    U, pu = stats.mannwhitneyu(v[ad], v[ctrl], alternative="two-sided")
    snpsg = snp_gene[snp_gene.gene == g]
    lead = snpsg.sort_values("P").iloc[0]
    rows.append(dict(gene=g, probe=pr.probe, log2FC_unadj=pr.log2FC_unadj,
                     p_ttest=pr.p_ord_unadj_ttest, log2FC_adj=pr.log2FC_adj,
                     p_mod_adj=pr.p_mod_adj, BH_adj=pr.padj_BH_mod_adj,
                     MW_U=U, MW_p=pu, n_supporting_SNPs=len(snpsg),
                     lead_SNP=lead.SNP, lead_SNP_P=lead.P, lead_chr=lead.CHR,
                     lead_pos_GRCh37=lead.BP_GRCh37, min_distance=snpsg.distance.min(),
                     in_both_lists_adj=g in cand, in_both_lists_unadj=g in cand_unadj))
cs = pd.DataFrame(rows)
cs.to_csv(os.path.join(OUT_DIR, "candidate_statistics_table.csv"), index=False)
P(cs.round(4).to_string())

# ------------------------------------------------ STEP 8: locus grouping (position only; NOT LD)
cl = cs[["gene", "lead_chr", "lead_pos_GRCh37"]].sort_values(["lead_chr", "lead_pos_GRCh37"])
cl = cl.reset_index(drop=True)
cl["locus"] = ((cl.lead_chr != cl.lead_chr.shift()) | (cl.lead_pos_GRCh37.diff() > 500_000)).cumsum()
P("Regions (lead SNPs >500 kb apart = separate region):\n", cl.to_string())

# ------------------------------------------------ STEP 9: threshold-free test (what intersection adds)
gl = de.dropna(subset=["gene"]).sort_values("p_mod_adj").drop_duplicates("gene").set_index("gene")
inG = gl.index.isin(gwas_genes); inN = gl.index.isin(gwas_noapoe)
for lab, m in (("all GWAS genes", inG), ("APOE region excluded", inN)):
    u, pu = stats.mannwhitneyu(gl.p_mod_adj[m], gl.p_mod_adj[~m], alternative="less")
    P(f"[competitive, threshold-free] {lab}: n={m.sum()} median p {np.median(gl.p_mod_adj[m]):.3f} "
      f"vs {np.median(gl.p_mod_adj[~m]):.3f}; one-sided MWU p={pu:.4f}")

# ------------------------------------------------ STEP 10: all EC samples, Braak ordinal (R4-4)
alls = meta.copy()
Ya = mat[alls.gsm].values.astype(float)
Xa = np.column_stack([np.ones(len(alls)), alls.braak.values, alls.sex01.values])
ba, ta, pa, _, _, _ = moderated_fit(Ya, Xa, 1)
dea = pd.DataFrame({"probe": probes_tested, "gene": gene_of_probe.values, "slope_per_braak_step": ba,
                    "t": ta, "p": pa, "BH": bh(pa)})
dea.to_csv(os.path.join(OUT_DIR, "S5_all_EC_samples_braak_ordinal.csv"), index=False)
P(f"All-EC model: n={len(alls)} samples; BH<0.05 probes = {(dea.BH < 0.05).sum()}")
P("Candidates in all-EC model (p, slope):\n",
  dea[dea.gene.isin(cand)].sort_values("p")[["gene", "slope_per_braak_step", "p"]].round(4).to_string())

# ------------------------------------------------ STEP 11: ranked gene-set analysis (prerank)
import gseapy as gp
rank = gl["t_mod_adj"].sort_values(ascending=False)
rank.index = rank.index.astype(str)
for lib in ("KEGG_2021_Human", "GO_Biological_Process_2023"):
    try:
        pre = gp.prerank(rnk=rank, gene_sets=lib, min_size=10, max_size=500,
                         permutation_num=1000, seed=SEED, outdir=None, no_plot=True)
        full = pre.res2d.sort_values("FDR q-val")
        full.to_csv(os.path.join(OUT_DIR, f"prerank_{lib}_FULL.csv"), index=False)
        r = full.head(15)
        P(lib, "\n", r[["Term", "NES", "NOM p-val", "FDR q-val"]].to_string())
    except Exception as e:
        P("prerank failed for", lib, e)

# ------------------------------------------------ STEP 12: enrichment with explicit background (R1-6, R3-5)
for label, gl_ in (("candidates", cand), ("all adjusted DEGs", sorted(deg_adj & universe))):
    for bg_label, bg in (("tested_genes_background", sorted(universe)), ("default_background", None)):
        try:
            kw = dict(gene_list=gl_, gene_sets=["KEGG_2021_Human", "GO_Biological_Process_2023"], outdir=None)
            if bg is not None: kw["background"] = bg
            res = gp.enrichr(**kw).results.sort_values("Adjusted P-value")
            res.to_csv(os.path.join(OUT_DIR, f"enrichr_{label.replace(' ', '_')}_{bg_label}.csv"), index=False)
            cols = [c for c in ["Gene_set", "Term", "Overlap", "P-value", "Adjusted P-value", "Genes"]
                    if c in res.columns]
            P(label, bg_label, "- columns:", list(res.columns))
            P(res[cols].head(10).to_string())
        except Exception as ex:
            P("enrichr failed:", label, bg_label, repr(ex))
P("DONE. Outputs in", OUT_DIR)

GRCh37 genes with symbols: 57905
Unique significant SNPs: 2394 (GRCh37 coordinates, Jansen 2019)
SNPs mapped: 2375 | GWAS genes: 128
rs429358 -> [['APOE', 0]]
rs7412 -> [['APOE', 0]]
Protein-coding-only GWAS genes: 97
SNPs mapped inside a gene (distance 0): 1325; mapped to nearest gene within window: 1050; SNPs assigned to >1 overlapping gene: 265
Biotypes of the GWAS genes:
 biotype
protein_coding          90
pseudogene              11
antisense                9
lincRNA                  5
snoRNA                   4
processed_transcript     3
miRNA                    3
snRNA                    2
misc_RNA                 1
status
no_assignment           283444
mapped                   23810
no_valid_symbol           5753
ambiguous_multi_gene      3912
Cytoband-like symbols remaining (must be 0): 0
Metadata columns available: ['subject id', 'tissue', 'tissue region', 'sex', 'braak nft stage']
Age/PMI/RIN-like fields found: ['braak nft stage']
EC samples by Braak: {0: 13, 1: 20, 2: 19, 3:

candidates tested_genes_background - columns: ['Gene_set', 'Term', 'P-value', 'Adjusted P-value', 'Old P-value', 'Old adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes']
                      Gene_set                                                                        Term   P-value  Adjusted P-value    Genes
1   GO_Biological_Process_2023                                          Enamel Mineralization (GO:0070166)  0.000635          0.012698  SLC24A4
2   GO_Biological_Process_2023                                           Tooth Mineralization (GO:0034505)  0.000847          0.012698  SLC24A4
3   GO_Biological_Process_2023                                 Regulation Of Feeding Behavior (GO:0060259)  0.001904          0.019041  SLC24A4
4   GO_Biological_Process_2023                                        Membrane Repolarization (GO:0086009)  0.003383          0.025375  SLC24A4
15  GO_Biological_Process_2023                  Calcium Ion Transmembrane Import Into Cytosol (GO:0097

DONE. Outputs in C:\Users\a\AlzheimerResearch\revision_outputs


In [4]:
"""Figure 3 at 300 dpi: PCA of the 32 analytical samples, colour = group, marker = sex.
Run in the same data folder as revision_analysis.py."""
import os, re
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

DATA_DIR = r"C:\Users\a\AlzheimerResearch"
gse = os.path.join(DATA_DIR, "GSE131617-GPL5175_series_matrix.txt")
gsm, titles, chars = [], [], []
with open(gse) as f:
    for line in f:
        if line.startswith("!Sample_geo_accession"): gsm = re.findall(r'"(GSM\d+)"', line)
        elif line.startswith("!Sample_title"): titles = re.findall(r'"([^"]*)"', line)
        elif line.startswith("!Sample_characteristics_ch1"): chars.append(re.findall(r'"([^"]*)"', line))
        elif line.startswith('"ID_REF"') or line.startswith("ID_REF"): break
meta = pd.DataFrame({"gsm": gsm, "title": titles})
for row in chars:
    keys = {x.split(":")[0].strip().lower() for x in row if ":" in x}
    if len(keys) == 1:
        meta[keys.pop()] = [x.split(":", 1)[1].strip() for x in row]
meta = meta[meta.title.str.contains("EC")]
meta = meta[meta["braak nft stage"].str.strip().isin(["0", "V-VI"])].copy()
meta["grp"] = np.where(meta["braak nft stage"].str.strip() == "0", "Control", "AD")
meta["sex"] = np.where(meta["sex"].str.upper().str.startswith("M"), "Male", "Female")
mat = pd.read_csv(gse, sep="\t", comment="!", header=0).set_index("ID_REF")
Y = mat[meta.gsm].values.astype(float)
Z = (Y - Y.mean(1, keepdims=True)) / Y.std(1, keepdims=True)
Z = Z[np.isfinite(Z).all(1)]
pca = PCA(n_components=2); pc = pca.fit_transform(Z.T)
fig, ax = plt.subplots(figsize=(6.4, 5.0))
col = {"Control": "#2C5FB3", "AD": "#C8372D"}; mk = {"Male": "o", "Female": "^"}
for g in col:
    for s in mk:
        m = ((meta.grp == g) & (meta.sex == s)).values
        ax.scatter(pc[m, 0], pc[m, 1], c=col[g], marker=mk[s], s=75, edgecolors="black", lw=0.8,
                   alpha=0.85, label=f"{g}, {s.lower()} (n={m.sum()})")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25, lw=0.5)
for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.tight_layout()
out = os.path.join(DATA_DIR, "revision_outputs", "Figure3_PCA_300dpi.png")
fig.savefig(out, dpi=300, facecolor="white")
print("PC1/PC2 variance:", pca.explained_variance_ratio_[:2], "->", out)

PC1/PC2 variance: [0.30425496 0.14665944] -> C:\Users\a\AlzheimerResearch\revision_outputs\Figure3_PCA_300dpi.png
